# MultiModel Testing with AutoGluon

## 1. Setup and Dependencies

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import torch
import random
from tqdm.auto import tqdm
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import sys

if os.path.basename(os.getcwd()) == 'notebooks':
    project_root = os.path.abspath('..')
else:
    project_root = os.getcwd()

if project_root not in sys.path:
    sys.path.append(project_root)

from src.datamodule import masked_smoothed_smape

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"AutoGluon TimeSeriesPredictor imported successfully")

## 2. Configuration

In [ ]:
BASE_DIR = ".."
DATA_DIR = os.path.join(BASE_DIR, "data")
TRAIN_DIR_FILTERED = os.path.join(DATA_DIR, "train")
VAL_DIR_FILTERED = os.path.join(DATA_DIR, "val")
# TRAIN_DIR_FILTERED = os.path.join(DATA_DIR, "train_trading_only")
# VAL_DIR_FILTERED = os.path.join(DATA_DIR, "val_trading_only")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODELS_DIR = os.path.join(BASE_DIR, "models")
AUTOGLUON_DIR = os.path.join(MODELS_DIR, "autogluon_quick_test") # Unique directory for quick test run

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(AUTOGLUON_DIR, exist_ok=True)

# Model configuration
TARGET_COLS = ["high", "low", "close", "volume"]
OUTPUT_CHUNK_LENGTH = 10  # Prediction length
SEED = 827

# Load a subset of assets for faster experimentation
MAX_ASSETS = 2 # Reduce assets for performance
# REVISION: Reduce LIMIT_ROWS_PER_ASSET for a very quick test run
LIMIT_ROWS_PER_ASSET = 5000 # Reduce rows per asset for performance

## 3. Load and Convert Data to AutoGluon Format

AutoGluon expects a DataFrame with columns: `[timestamp, item_id, target]`

In [ ]:
def load_parquet_files(directory, max_assets=None):
    """Load all parquet files from directory."""
    all_files = [f for f in os.listdir(directory) if f.endswith('.parquet')]
    
    random.seed(SEED)
    random.shuffle(all_files)

    if max_assets:
        files = all_files[:max_assets]
    else:
        files = all_files
    
    data = {}
    for file in tqdm(files, desc="Loading parquet files"):
        asset_name = file.replace('.parquet', '')
        df = pd.read_parquet(os.path.join(directory, file))
        data[asset_name] = df
    
    return data


def convert_to_autogluon_format(data_dict, target_cols, limit_rows=None, dataset_name="data"):
    """
    Convert dict of asset DataFrames to AutoGluon TimeSeriesDataFrame format.
    Includes all other columns (except target and ExecutionTime) as past_covariates.
    
    Per AutoGluon documentation:
    - Columns in train_data except 'target' and 'known_covariates_names' are interpreted as past_covariates
    - known_covariates_names should be empty (we have no future covariates)
    - tuning_data should be removed from fit() to allow proper covariate interpretation
    """
    all_data = []
    
    # List of columns to exclude from covariates
    exclude_cols = target_cols + ['ExecutionTime']

    for asset_name, df in tqdm(data_dict.items(), desc="Converting to AutoGluon format"):
        if limit_rows:
            df = df.iloc[-limit_rows:].copy()
        else:
            df = df.copy()
            
        # Clean and prepare timestamps
        df['ExecutionTime'] = pd.to_datetime(df['ExecutionTime'])
        if df['ExecutionTime'].dt.tz is not None:
            df['ExecutionTime'] = df['ExecutionTime'].dt.tz_localize(None)
        
        # Identify covariate columns (all except target and ExecutionTime)
        covariate_cols = [c for c in df.columns if c not in exclude_cols]
        
        # Ensure 'is_trading' is treated as a float for consistency
        if 'is_trading' in df.columns:
            df['is_trading'] = df['is_trading'].astype(np.float32)

        for col in target_cols:
            if col in df.columns:
                # Create the base DataFrame for this target
                item_df = pd.DataFrame({
                    'timestamp': df['ExecutionTime'].values, 
                    'item_id': f"{asset_name}_{col}",  # Unique ID per asset-feature
                    'target': df[col].astype(np.float32).values
                })
                
                # Add all identified covariate columns as past_covariates
                for cov_col in covariate_cols:
                    if cov_col in df.columns:
                        item_df[cov_col] = df[cov_col].values
                
                # Drop rows where target or any covariate is NaN
                item_df = item_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['target'] + covariate_cols)
                all_data.append(item_df)
    
    combined_df = pd.concat(all_data, ignore_index=True)
    
    print(f"\nCombined DataFrame shape: {combined_df.shape}")
    print(f"Columns: {combined_df.columns.tolist()}")
    
    # Convert to TimeSeriesDataFrame
    ts_df = TimeSeriesDataFrame.from_data_frame(
        combined_df,
        id_column='item_id',
        timestamp_column='timestamp'
    )
    
    # Do NOT set known_covariate_names - keep it empty
    # AutoGluon will automatically interpret extra columns as past_covariates
    print(f"\nPast covariates (will be auto-detected by AutoGluon): {covariate_cols}")
    print(f"Total covariates: {len(covariate_cols)}")

    return ts_df


print("Loading data...")
train_data = load_parquet_files(TRAIN_DIR_FILTERED, max_assets=MAX_ASSETS)
val_data = load_parquet_files(VAL_DIR_FILTERED, max_assets=MAX_ASSETS)

print(f"\nLoaded {len(train_data)} training assets")
print(f"Loaded {len(val_data)} validation assets")

# Convert to AutoGluon format
train_ts = convert_to_autogluon_format(train_data, TARGET_COLS, limit_rows=LIMIT_ROWS_PER_ASSET, dataset_name="train")
val_ts = convert_to_autogluon_format(val_data, TARGET_COLS, limit_rows=LIMIT_ROWS_PER_ASSET, dataset_name="val")

print(f"\n{'='*80}")
print("AutoGluon TimeSeriesDataFrame Created (with Past Covariates)")
print(f"{'='*80}")
print(f"Train shape: {train_ts.shape}")
print(f"Val shape: {val_ts.shape}")

## 4. Compare Global Models with AutoGluon

We'll use multiple hyperparameter configurations for Deep Learning models to simulate:
Zero-Shot (minimal train) vs Fine-Tuned.

In [ ]:
TIME_LIMIT = 1200

print("="*80)
print("AUTOGLUON GLOBAL MODEL COMPARISON (QUICK TEST)")
print("="*80)
print(f"Prediction length: {OUTPUT_CHUNK_LENGTH}")
print(f"Time limit: {TIME_LIMIT} seconds ({TIME_LIMIT/60:.1f} minutes) for test run") 
print(f"GPU: {'Available' if torch.cuda.is_available() else 'Not available'}")
print("="*80)

# Define hyperparameters for a fast test run
hyperparameters = {
    # 1. DeepAR (Simulating Zero-Shot/Fine-Tuning)
    "DeepAR": [
        {"max_epochs": 1, "ag_args": {"name_suffix": "_ZeroShotBase"}},
        {"max_epochs": 10, "ag_args": {"name_suffix": "_FineTuned"}},
    ],
    # 2. Temporal Fusion Transformer (TFT) (Simulating Zero-Shot/Fine-Tuning)
    "TemporalFusionTransformer": [
        {"max_epochs": 1, "ag_args": {"name_suffix": "_ZeroShotBase"}},
        {"max_epochs": 10, "ag_args": {"name_suffix": "_FineTuned"}},
    ],
    # 3. PatchTST (Modern Transformer Model)
    "PatchTST": [
        {"max_epochs": 1, "ag_args": {"name_suffix": "_ZeroShotBase"}},
        {"max_epochs": 10, "ag_args": {"name_suffix": "_FineTuned"}},
    ],
    # 4. DirectTabular (PerStepTabularModel - Non-Deep Learning Baseline)
    "DirectTabular": {
        "ag_args": {"name_suffix": "TabularBaseline"}
    }
}

print("\nStarting training...")
print("\nIMPORTANT: Not providing tuning_data to fit() so AutoGluon can")
print("properly interpret extra columns as past_covariates.\n")

predictor = TimeSeriesPredictor(
    prediction_length=OUTPUT_CHUNK_LENGTH,
    path=AUTOGLUON_DIR,
    target="target",
    eval_metric="MASE",
    freq="15min",
    verbosity=2
)

# CRITICAL: Do NOT provide tuning_data here
# Per documentation: tuning_data should be removed so extra columns 
# in train_data are properly interpreted as past_covariates
predictor.fit(
    train_data=train_ts,
    hyperparameters=hyperparameters,
    time_limit=TIME_LIMIT,
    random_seed=SEED
)

print("\n✓ Training complete!")

## 5. Evaluate Models

In [ ]:
print("="*80)
print("MODEL EVALUATION (MASE - AutoGluon Metric)")
print("="*80)

# Get leaderboard (since we didn't provide tuning_data, leaderboard will show train results)
leaderboard = predictor.leaderboard(silent=False)
print("\nLeaderboard (Ranked by MASE):")
print(leaderboard)

# Get best model info
best_model = leaderboard.iloc[0]['model']
best_score = leaderboard.iloc[0]['score_val']

print(f"\nBest Model (by MASE): {best_model}, Score (MASE): {best_score:.4f}")

# Optionally evaluate on validation data
print("\n" + "="*80)
print("VALIDATION SET EVALUATION")
print("="*80)
val_predictions = predictor.predict(val_ts)
print(f"✓ Generated predictions on validation set: {val_predictions.shape}")

## 7. Save Final Results

In [ ]:
# Compile results
results = {
    'method': 'AutoGluon TimeSeriesPredictor (Covariates Enabled)',
    'model': 'DeepAR, TFT, PatchTST, DirectTabular Comparison',
    'prediction_length': OUTPUT_CHUNK_LENGTH,
    'best_model_mase': best_model,
    'best_score_mase': float(best_score),
    'num_assets': MAX_ASSETS,
    'num_features': len(TARGET_COLS),
    'num_items': len(train_ts.item_ids.unique()),
    'leaderboard_mase': leaderboard.to_dict('records')
}

# Save results
results_file = os.path.join(RESULTS_DIR, "autogluon_results_test_covariates.json")
with open(results_file, 'w') as f:
    json.dump(results, f, indent=4)

print("="*80)
print("RESULTS SAVED")
print(f"Results: {results_file}")
print("="*80)

In [ ]:
# Load the saved predictor
loaded_predictor = TimeSeriesPredictor.load(AUTOGLUON_DIR)

print("✓ Model loaded successfully")
print(f"\nModel info:")
print(f"  Path: {loaded_predictor.path}")
print(f"  Prediction length: {loaded_predictor.prediction_length}")
print(f"  Target: {loaded_predictor.target}")

# Make predictions with loaded model
test_predictions = loaded_predictor.predict(val_ts.head(100))
print(f"\nTest predictions shape: {test_predictions.shape}")
print("✓ Loaded model works correctly")